# Publication figures — shipwreck deterioration

Generate manuscript Figures 2–6 and Supplementary Figures S1–S12 from the final analysis products. Figure 1 is prepared separately in PowerPoint.

Run cells in order. Export settings and filenames are defined in the final section.


## 0. Packages, paths and plotting theme


In [ ]:
library(arrow)
library(mgcv)
library(ggplot2)
library(terra)
library(tidyterra)
library(sf)
library(rnaturalearth)
library(rnaturalearthdata)
library(data.table)
library(cowplot)
library(grid)

fig <- function(height, width) {
  options(repr.plot.width = width, repr.plot.height = height)
}

plot_theme1 <- theme(
  strip.text = element_text(size = 16, face = "bold"),
  strip.background = element_blank(),
  axis.text = element_text(size = 18, colour = "black"),
  axis.title = element_text(size = 22),
  panel.grid = element_blank(),
  legend.position = "bottom",
  legend.title = element_text(size = 16, face = "bold"),
  legend.text = element_text(size = 14)
)

project_dir <- normalizePath("..")

ssd_dir <- "/media/syms/ExtremeSSD/ShipwreckENDURE"
data_dir <- file.path(ssd_dir, "data")
figmaps_dir <- file.path(ssd_dir, "outputs", "figure_maps")
driver_dir <- file.path(ssd_dir, "outputs", "driver_maps")

figure_data_dir <- file.path(project_dir, "outputs", "figure_data")

figure_out_dir <- file.path(project_dir, "outputs", "manuscript_figures")
dir.create(figure_out_dir, recursive = TRUE, showWarnings = FALSE)

world <- ne_countries(scale = "large", returnclass = "sf")

cat("Project:", project_dir, "\n")
cat("Figure data:", figure_data_dir, "\n")
cat("Figure output:", figure_out_dir, "\n")
cat("SSD model products:", ssd_dir, "\n")


## 1. Calibration model and shared data

Calibration objects used in Figure 2 and Supplementary Figures S1–S3.


In [ ]:
input_file <- file.path(ssd_dir, "wreck_agent_decay_for_gam.parquet")
curve_file <- file.path(data_dir, "gam_age_curve.csv")

if (!file.exists(input_file)) stop("Missing calibration input: ", input_file)
if (!dir.exists(data_dir)) dir.create(data_dir, recursive = TRUE)


In [ ]:
df <- data.frame(read_parquet(input_file))

required <- c(
  "JD.Contractor.Listing",
  "Obs",
  "chronological_age",
  "log_CR_physics",
  "log_CR_bio"
)

missing <- setdiff(required, names(df))
if (length(missing) > 0) {
  stop("Missing required columns: ", paste(missing, collapse = ", "))
}

cal <- df[
  complete.cases(df[, required]) &
    df$Obs > 0,
]

cal$Obs <- as.integer(cal$Obs)

cat("Rows from Notebook 04:", nrow(df), "\n")
cat("Rows used for GAM calibration:", nrow(cal), "\n")
cat("Age range:", paste(range(cal$chronological_age), collapse = " to "), "\n")
cat("Observed classes:", paste(sort(unique(cal$Obs)), collapse = ", "), "\n")

if (any(cal$chronological_age <= 0)) {
  stop("Non-positive chronological age detected.")
}

expected_classes <- seq_len(length(unique(cal$Obs)))
if (!identical(sort(unique(cal$Obs)), expected_classes)) {
  stop("Observed Muckelroy classes are not consecutive integers starting at 1.")
}


In [ ]:
Rcat <- length(unique(cal$Obs))

model1 <- gam(
  Obs ~
    s(chronological_age, k = 5) +
    s(log_CR_physics, k = 5) +
    s(log_CR_bio, k = 5),
  family = ocat(R = Rcat),
  data = cal,
  method = "REML"
)

model2 <- gam(
  Obs ~
    s(chronological_age, k = 5) +
    s(log_CR_physics, k = 5),
  family = ocat(R = Rcat),
  data = cal,
  method = "REML"
)

model4 <- gam(
  Obs ~ s(chronological_age, k = 5),
  family = ocat(R = Rcat),
  data = cal,
  method = "REML"
)

summary(model1)
AIC(model1, model2, model4)

gam.check(model1)
gam.check(model2)
gam.check(model4)


In [ ]:
age_compare <- seq(
  min(cal$chronological_age),
  max(cal$chronological_age),
  length.out = 300
)

mean_phys <- mean(cal$log_CR_physics, na.rm = TRUE)
mean_bio <- mean(cal$log_CR_bio, na.rm = TRUE)

new_m1 <- data.frame(
  chronological_age = age_compare,
  log_CR_physics = mean_phys,
  log_CR_bio = mean_bio
)

new_m2 <- data.frame(
  chronological_age = age_compare,
  log_CR_physics = mean_phys
)

new_m4 <- data.frame(
  chronological_age = age_compare
)

expected_class <- function(model, newdata) {
  p <- predict(model, newdata = newdata, type = "response")
  as.numeric(p %*% seq_len(ncol(p)))
}

e1 <- expected_class(model1, new_m1)
e2 <- expected_class(model2, new_m2)
e4 <- expected_class(model4, new_m4)

fig(6, 8)
plot(
  age_compare, e4,
  type = "l",
  lwd = 2,
  ylim = range(c(e1, e2, e4)),
  xlab = "Chronological age (years)",
  ylab = "Expected Muckelroy condition class"
)

lines(age_compare, e2, lwd = 2, lty = 2)
lines(age_compare, e1, lwd = 2, lty = 3)

legend(
  "bottomright",
  legend = c(
    "Age only",
    "Age + physical CR",
    "Age + physical + biological CR"
  ),
  lty = c(1, 2, 3),
  lwd = 2,
  bty = "n"
)


In [ ]:
age_quantiles <- quantile(
  cal$chronological_age,
  probs = c(0.10, 0.90, 0.95, 0.99)
)

print(age_quantiles)

support_age_max <- 125
tail_lambda0 <- 0.01

cat("Production support age:", support_age_max, "years\n")
cat("Wrecks older than support age:",
    sum(cal$chronological_age > support_age_max), "\n")


In [ ]:
age_grid <- seq(0, 300, by = 1)

newdata <- data.frame(
  chronological_age = age_grid,
  log_CR_physics = mean_phys,
  log_CR_bio = mean_bio
)

raw_pred <- expected_class(model1, newdata)

mono_pred <- cummax(raw_pred)
model_pred <- mono_pred

support_idx <- which(age_grid <= support_age_max)
tail_idx <- which(age_grid > support_age_max)

class_at_support <- mono_pred[max(support_idx)]

model_pred[tail_idx] <- 4 - (4 - class_at_support) *
  exp(-tail_lambda0 * (age_grid[tail_idx] - support_age_max))

model_pred <- pmin(model_pred, 4)

gam_curve_out <- data.frame(
  Age = age_grid,
  Raw_GAM = raw_pred,
  Monotonic_GAM = mono_pred,
  ExpectedClass = model_pred
)

if (nrow(gam_curve_out) != 301) stop("Unexpected gam_age_curve row count.")
if (!identical(names(gam_curve_out),
               c("Age", "Raw_GAM", "Monotonic_GAM", "ExpectedClass"))) {
  stop("Unexpected gam_age_curve column structure.")
}
if (any(diff(gam_curve_out$ExpectedClass) < 0)) {
  stop("Production ExpectedClass curve is not monotonic.")
}

write.csv(
  gam_curve_out,
  file = curve_file,
  row.names = FALSE
)

cat("Saved:", curve_file, "\n")
cat("Expected class at 125 years:",
    gam_curve_out$ExpectedClass[gam_curve_out$Age == 125], "\n")

head(gam_curve_out)
tail(gam_curve_out)


In [ ]:
figure_age <- seq(
  min(cal$chronological_age, na.rm = TRUE),
  250,
  by = 1
)

figure_newdata <- data.frame(
  chronological_age = figure_age,
  log_CR_physics = mean_phys,
  log_CR_bio = mean_bio
)

figure_raw <- expected_class(model1, figure_newdata)
figure_mono <- cummax(figure_raw)
figure_model <- figure_mono

support_idx_fig <- which(figure_age <= support_age_max)
tail_idx_fig <- which(figure_age > support_age_max)

class_at_support_fig <- figure_mono[max(support_idx_fig)]

figure_model[tail_idx_fig] <- 4 - (4 - class_at_support_fig) *
  exp(-tail_lambda0 * (figure_age[tail_idx_fig] - support_age_max))

figure_model <- pmin(figure_model, 4)

curve_df <- data.frame(
  Age = figure_age,
  Raw_GAM = figure_raw,
  Monotonic_GAM = figure_mono,
  Model_curve = figure_model
)

set.seed(123)

B <- 300
pred_mat <- matrix(
  NA_real_,
  nrow = length(figure_age),
  ncol = B
)

for (b in seq_len(B)) {
  idx <- sample(seq_len(nrow(cal)), replace = TRUE)
  dat_b <- cal[idx, ]

  fit_b <- gam(
    Obs ~
      s(chronological_age, k = 5) +
      s(log_CR_physics, k = 5) +
      s(log_CR_bio, k = 5),
    family = ocat(R = Rcat),
    data = dat_b,
    method = "REML"
  )

  pred_mat[, b] <- expected_class(fit_b, figure_newdata)
}

curve_df$Raw_GAM_lwr <- apply(
  pred_mat, 1, quantile, 0.025, na.rm = TRUE
)
curve_df$Raw_GAM_upr <- apply(
  pred_mat, 1, quantile, 0.975, na.rm = TRUE
)

raw_plot_df <- curve_df[
  curve_df$Age <= max(cal$chronological_age, na.rm = TRUE),
]


## 2. Main manuscript figures


### Figure 2 — Calibration: chronological age and Muckelroy condition


In [ ]:
final_figure <- ggplot() +
  geom_ribbon(
    data = raw_plot_df,
    aes(
      x = Age,
      ymin = Raw_GAM_lwr,
      ymax = Raw_GAM_upr
    ),
    alpha = 0.2
  ) +
geom_jitter(
    data = cal,
    aes(x = chronological_age, y = Obs),
    width = 0,
    height = 0.06,
    alpha = 0.45,
    color = "black",
    size = 2
  ) +
  geom_line(
    data = raw_plot_df,
    aes(x = Age, y = Raw_GAM),
    linewidth = 1,
    linetype = "dashed"
  ) +
  geom_line(
    data = curve_df,
    aes(x = Age, y = Model_curve),
    linewidth = 1.4
  ) +
  geom_vline(
    xintercept = support_age_max,
    linetype = "dotted"
  ) +
  geom_rug(
    data = cal,
    aes(x = chronological_age, y = Obs),
    sides = "b"
  ) +
  coord_cartesian(ylim = c(0.5, 4.1)) +
  xlab("Chronological age (years)") +
  ylab("Muckelroy class") +
  theme_classic() +
  theme(
    plot.title = element_blank(),
    axis.text = element_text(size = 14),
    axis.title = element_text(size = 14),
    panel.background = element_blank(),
    panel.grid = element_blank(),
    axis.line = element_line(colour = "black")
  )

final_figure


### Figure 3 — Environmental and mechanical-forcing drivers


In [ ]:
sal <- rast(file.path(driver_dir, "salinity_mean.tif"))
temp <- rast(file.path(driver_dir, "bottom_temperature_mean.tif"))
mechanical <- rast(file.path(driver_dir, "mechanical_forcing_modifier.tif"))

sites <- data.frame(
  site = c(
    "Baltic refuge", "Danish Straits", "Jutland shelf",
    "Southern North Sea", "Norwegian trench", "Baltic entrance"
  ),
  latitude = c(56.5, 56.3, 56.4, 54.8, 57.35, 55),
  longitude = c(18.0, 11.5, 6.4, 6.5, 7.7, 13)
)

sites_sf <- st_as_sf(
  sites,
  coords = c("longitude", "latitude"),
  crs = 4326
)

base_driver_map <- function(x) {
  ggplot(world) +
    theme_bw() +
    geom_spatraster(data = x) +
    geom_sf() +
    geom_sf(
      data = sites_sf,
      colour = "black", fill = "black",
      shape = 21, size = 3
    ) +
    coord_sf(
      xlim = c(xmin(x), xmax(x)),
      ylim = c(ymin(x), ymax(x)),
      expand = FALSE
    ) +
    theme(
      axis.text = element_text(size = 16, colour = "black"),
      panel.background = element_rect(fill = "lightblue1", colour = "lightblue1"),
      panel.grid.major = element_line(linewidth = 0.5, linetype = "solid", colour = "grey")
    )
}

templot <- base_driver_map(temp) +
  scale_fill_gradientn(
    colours = c("#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"),
    name = "Temperature\n(°C)\n"
  ) +
  labs(title = "(a) Seafloor temperature")

salplot <- base_driver_map(sal) +
  scale_fill_distiller(
    palette = "RdYlBu",
    direction = 1,
    name = "Salinity\n"
  ) +
  labs(title = "(b) Salinity")

hydroplot <- base_driver_map(mechanical) +
  scale_fill_gradientn(
    colours = c("white", "lightblue", "royalblue", "navy"),
    name = "Forcing\nstrength\n",
    limits = c(0, 1),
    oob = scales::squish
  ) +
  labs(title = "(c) Mechanical forcing")

driver_title_theme <- theme(
  plot.title = element_text(hjust = 0, face = "bold", size = 22),
  legend.title = element_text(size = 12),
  legend.text = element_text(size = 10),
  plot.margin = margin(8, 5, 8, 5)
)

templot <- templot + driver_title_theme
salplot <- salplot + driver_title_theme
hydroplot <- hydroplot + driver_title_theme

fig(15, 12)
driver_maps <- cowplot::plot_grid(
  templot, salplot, hydroplot,
  ncol = 1,
  align = "v",
  axis = "lr"
)
driver_maps


### Figure 4 — Parameter sensitivity


In [ ]:
parameter_labels <- c(
  Ea = "Activation energy",
  U_current_ref = "Current reference velocity",
  U_wave_ref = "Wave reference velocity",
  hydro_interaction_strength = "Current–wave interaction",
  S_mid_corr = "Salinity midpoint",
  burial_alpha = "Burial exponent",
  burial_strength = "Burial strength",
  k_teredo_high = "Teredo intensity"
)

tornado <- fread(file.path(
  figure_data_dir,
  "fig5_tornado_summary.csv"
))

tornado[, label := parameter_labels[parameter]]
tornado[, low_pct := 100 * (low - 1)]
tornado[, high_pct := 100 * (high - 1)]
tornado[, max_abs_pct := pmax(abs(low_pct), abs(high_pct))]

tornado <- tornado[order(max_abs_pct)]
tornado[, label := factor(label, levels = label)]

p_sensitivity <- ggplot(tornado, aes(y = label)) +
  geom_segment(
    aes(x = low_pct, xend = high_pct, yend = label),
    linewidth = 5,
    lineend = "butt"
  ) +
  geom_vline(xintercept = 0, linetype = "dotted") +
  labs(
    x = "Change from baseline median rate scale (%)",
    y = NULL
  ) +
  theme_classic() +
  plot_theme1

fig(7, 10)
p_sensitivity


### Figure 5 — Regional time to Muckelroy class 3


In [ ]:
steel <- rast(file.path(figmaps_dir, "steel_none_time_to_muckelroy_3.tif"))
woodnone <- rast(file.path(figmaps_dir, "wood_none_time_to_muckelroy_3.tif"))
woodhigh <- rast(file.path(figmaps_dir, "wood_high_time_to_muckelroy_3.tif"))

threshold_limits <- range(
  c(values(steel), values(woodnone), values(woodhigh)),
  na.rm = TRUE
)

r <- steel

map_threshold <- function(x, title) {
  ggplot(world) +
    theme_bw() +
    labs(title = title) +
    geom_spatraster(data = x) +
    geom_sf() +
    coord_sf(
      xlim = c(xmin(r), xmax(r)),
      ylim = c(ymin(r), ymax(r)),
      expand = FALSE
    ) +
    scale_fill_viridis_c(
      name = "Years to\nMuckelroy 3\n",
      limits = threshold_limits,
      oob = scales::squish
    ) +
    theme(
      axis.text = element_text(size = 16, colour = "black"),
      panel.background = element_rect(fill = "lightblue1", colour = "lightblue1"),
      panel.grid.major = element_line(linewidth = 0.5, linetype = "solid", colour = "grey"),
      plot.title = element_text(hjust = 0, face = "bold", size = 22),
      legend.title = element_text(size = 12),
      legend.text = element_text(size = 10),
      plot.margin = margin(8, 5, 8, 5)
    )
}

steelplot <- map_threshold(steel, "(a) Steel wrecks")
woodplot <- map_threshold(woodnone, "(b) Wood wrecks, no Teredo")
teredoplot <- map_threshold(woodhigh, "(c) Wood wrecks, high Teredo")

fig(15, 12)
threshold_maps <- cowplot::plot_grid(
  steelplot, woodplot, teredoplot,
  ncol = 1,
  align = "v",
  axis = "lr"
)
threshold_maps


### Figure 6 — Representative condition trajectories


In [ ]:
traj <- fread(file.path(
  figure_data_dir,
  "representative_site_trajectories.csv"
))

traj[, site := factor(
  site,
  levels = c(
    "Norwegian trench",
    "Danish Straits",
    "Jutland shelf",
    "Baltic entrance",
    "Southern North Sea",
    "Baltic"
  )
)]

traj[, scenario := factor(
  scenario,
  levels = c("Steel", "Wood_none", "Wood_high"),
  labels = c("Steel", "Wood, No Teredo", "Wood, High Teredo")
)]

p_traj <- ggplot(
  traj,
  aes(
    x = chronological_age,
    y = predicted_condition,
    linetype = scenario
  )
) +

  geom_line(linewidth = 1.2) +
  scale_linetype_manual(
    values = c(
      "Steel" = "solid",
      "Wood, No Teredo" = "dashed",
      "Wood, High Teredo" = "dotted"
    )
  ) +
  facet_wrap(~ site, ncol = 2) +
  geom_hline(yintercept = 3, linetype = "dotted") +
  coord_cartesian(ylim = c(1, 4), xlim = c(0, 300)) +
  labs(
    x = "Chronological age (years)",
    y = "Predicted Muckelroy condition class",
    linetype = "Scenario"
  ) +
  theme_classic() +
  plot_theme1 +
  theme(
    legend.key.width = unit(2.5, "cm")
  )

fig(10, 12)
p_traj


## 3. Supplementary figures


### Figure S1 — Wreck age and Muckelroy-class calibration data


In [ ]:
p_age_class <- ggplot(
  cal,
  aes(x = as.factor(Obs), y = chronological_age)
) +
  geom_boxplot(
    alpha = 0.4,
    outlier.shape = NA
  ) +
  geom_jitter(
    width = 0.2,
    height = 0,
    size = 2,
    alpha = 0.7,
    color = "black"
  ) +
  labs(
    x = "Muckelroy class",
    y = "Age"
  ) +
  theme_classic() +
  plot_theme1

p_age_class


In [ ]:
# support_age_max marks the 125-year empirical support threshold used by the model.
p_wreck_counts <- ggplot(
  cal,
  aes(x = chronological_age)
) +
  geom_histogram(
    bins = 10,
    fill = "grey80",
    color = "black"
  ) +
  geom_vline(
    xintercept = support_age_max,
    linetype = "dotted"
  ) +
  scale_y_continuous(expand = c(0, 0)) +
  labs(
    x = "Chronological age (years)",
    y = "Wreck count"
  ) +
  theme_classic() +
  plot_theme1

p_wreck_counts


### Figure S2 — GAM basis / k comparison


In [ ]:
age_basis_grid <- seq(
  min(cal$chronological_age, na.rm = TRUE),
  max(cal$chronological_age, na.rm = TRUE),
  length.out = 200
)

basis_newdata <- data.frame(
  chronological_age = age_basis_grid,
  log_CR_physics = mean(cal$log_CR_physics, na.rm = TRUE),
  log_CR_bio = mean(cal$log_CR_bio, na.rm = TRUE)
)

fit_basis_model <- function(bs, k) {
  gam(
    Obs ~
      s(chronological_age, k = k, bs = bs) +
      s(log_CR_physics, k = 5) +
      s(log_CR_bio, k = 5),
    family = ocat(R = Rcat),
    data = cal,
    method = "REML"
  )
}

basis_settings <- expand.grid(
  bs = c("tp", "cr"),
  k = c(5, 7),
  stringsAsFactors = FALSE
)

basis_pred_list <- vector("list", nrow(basis_settings))

for (i in seq_len(nrow(basis_settings))) {
  m <- fit_basis_model(basis_settings$bs[i], basis_settings$k[i])

  basis_pred_list[[i]] <- data.frame(
    chronological_age = age_basis_grid,
    expected_class = expected_class(m, basis_newdata),
    basis = basis_settings$bs[i],
    k = basis_settings$k[i],
    label = paste0(basis_settings$bs[i], ", k = ", basis_settings$k[i])
  )
}

basis_pred_df <- do.call(rbind, basis_pred_list)

p_basis <- ggplot(
  basis_pred_df,
  aes(
    x = chronological_age,
    y = expected_class,
    linetype = label,
    linewidth = label
  )
) +
  geom_line() +
  geom_vline(
    xintercept = support_age_max,
    linetype = "dotted"
  ) +
  scale_linetype_manual(
    values = c(
      "tp, k = 5" = "solid",
      "tp, k = 7" = "longdash",
      "cr, k = 5" = "dotted",
      "cr, k = 7" = "dotdash"
    )
  ) +
  scale_linewidth_manual(
    values = c(
      "tp, k = 5" = 1.3,
      "tp, k = 7" = 0.8,
      "cr, k = 5" = 1.3,
      "cr, k = 7" = 0.8
    ),
    guide = "none"
  ) +
  labs(
    x = "Chronological age (years)",
    y = "Expected condition class",
    linetype = "Basis / k"
  ) +
  theme_classic() +
  plot_theme1

p_basis


### Figure S3 — Observed versus condition-derived chronological age


In [ ]:
age_lookup <- curve_df[, c("Age", "Model_curve")]

cal$predicted_chronological_age <- NA_real_

for (i in seq_len(nrow(cal))) {
  obs_i <- cal$Obs[i]

  if (!is.na(obs_i)) {
    j <- which.min(abs(age_lookup$Model_curve - obs_i))
    cal$predicted_chronological_age[i] <- age_lookup$Age[j]
  }
}

p_age_validation <- ggplot(
  cal,
  aes(
    x = chronological_age,
    y = predicted_chronological_age
  )
) +
  geom_point(
    alpha = 0.45,
    color = "black",
    size = 2
  ) +
  geom_abline(
    intercept = 0,
    slope = 1,
    linetype = "dotted"
  ) +
  geom_smooth(
    method = "lm",
    formula = y ~ x,
    se = TRUE,
    color = "black",
    fill = "grey70",
    linewidth = 1.2
  ) +
  labs(
    x = "Observed chronological age (years)",
    y = "Predicted chronological age (years)"
  ) +
  theme_classic() +
  plot_theme1

p_age_validation


### Figure S4 — Arrhenius temperature response


In [ ]:
arrhenius <- function(T_C, Ea = 50000, T_ref_C = 20) {
  R <- 8.314
  T_K <- T_C + 273.15
  T_ref_K <- T_ref_C + 273.15
  exp((-Ea / R) * ((1 / T_K) - (1 / T_ref_K)))
}

T_seq <- seq(0, 30, length.out = 300)

arr_df <- data.frame(
  temperature = rep(T_seq, 3),
  modifier = c(
    arrhenius(T_seq, Ea = 30000),
    arrhenius(T_seq, Ea = 50000),
    arrhenius(T_seq, Ea = 70000)
  ),
  Ea = factor(
    rep(c("30", "50 baseline", "70"), each = length(T_seq)),
    levels = c("30", "50 baseline", "70")
  )
)

p_arrhenius <- ggplot(
  arr_df,
  aes(temperature, modifier, linetype = Ea, linewidth = Ea)
) +
  geom_line() +
  scale_linetype_manual(
    values = c("30" = "dashed", "50 baseline" = "solid", "70" = "longdash")
  ) +
  scale_linewidth_manual(
    values = c("30" = 0.8, "50 baseline" = 1.2, "70" = 0.8),
    guide = "none"
  ) +
  labs(
    x = "Temperature (°C)",
    y = "Relative modifier",
    linetype = "Activation energy (kJ mol-1)"
  ) +
  geom_hline(yintercept = 1, linetype = "dotted") +
  geom_vline(xintercept = 20, linetype = "dotted") +
  theme_classic() +
  plot_theme1

fig(7, 10)
p_arrhenius


### Figure S5 — Salinity response


In [ ]:
logistic_salinity <- function(
  S,
  S_mid = 14,
  S_steep = 6,
  f_min = 0.05,
  f_max = 1.0
) {
  f_min + (f_max - f_min) / (1 + exp(-(S - S_mid) / S_steep))
}

S_seq <- seq(0, 40, length.out = 300)

sal_df <- data.frame(
  salinity = rep(S_seq, 5),
  modifier = c(
    logistic_salinity(S_seq, S_mid = 8),
    logistic_salinity(S_seq, S_mid = 10),
    logistic_salinity(S_seq, S_mid = 12),
    logistic_salinity(S_seq, S_mid = 14),
    logistic_salinity(S_seq, S_mid = 16)
  ),
  S_mid = factor(
    rep(c("8", "10", "12", "14 baseline", "16"), each = length(S_seq)),
    levels = c("8", "10", "12", "14 baseline", "16")
  )
)

p_salinity_response <- ggplot(
  sal_df,
  aes(salinity, modifier, linetype = S_mid, linewidth = S_mid)
) +
  geom_line() +
  geom_hline(yintercept = 1, linetype = "dotted") +
  scale_linetype_manual(
    values = c(
      "8" = "dotted",
      "10" = "dotdash",
      "12" = "twodash",
      "14 baseline" = "solid",
      "16" = "dashed"
    )
  ) +
  scale_linewidth_manual(
    values = c(
      "8" = 0.8,
      "10" = 0.8,
      "12" = 0.8,
      "14 baseline" = 1.2,
      "16" = 0.8
    ),
    guide = "none"
  ) +
  labs(
    x = "Salinity",
    y = "Relative modifier",
    linetype = "Salinity midpoint (PSU)"
  ) +
  theme_classic() +
  plot_theme1

fig(7, 10)
p_salinity_response


### Figure S6 — Mechanical forcing response


In [ ]:
forcing_modifier <- function(U, U_ref) {
  U / (U + U_ref)
}

U_refs <- c(0.15, 0.30, 0.60)
ref_labels <- c("0.15", "0.30 (baseline)", "0.60")

ref_linetypes <- c(
  "0.15" = "dotted",
  "0.30 (baseline)" = "solid",
  "0.60" = "dashed"
)

ref_linewidths <- c(
  "0.15" = 0.8,
  "0.30 (baseline)" = 1.2,
  "0.60" = 0.8
)

forcing_theme <- theme(
  plot.title = element_text(
    size = 18,
    face = "bold",
    hjust = 0
  ),
  axis.title = element_text(
    size = 14
  ),
  axis.text = element_text(
    size = 11,
    colour = "black"
  ),
  legend.position = "bottom",
  legend.title = element_text(
    size = 13,
    face = "bold"
  ),
  legend.text = element_text(
    size = 12
  )
)

U <- seq(0, 0.7, length.out = 500)

current_df <- CJ(U = U, U_ref = U_refs)
current_df[, modifier := forcing_modifier(U, U_ref)]
current_df[, U_ref := factor(U_ref, levels = U_refs, labels = ref_labels)]

wave_df <- CJ(U = U, U_ref = U_refs)
wave_df[, modifier := forcing_modifier(U, U_ref)]
wave_df[, U_ref := factor(U_ref, levels = U_refs, labels = ref_labels)]

p_current <- ggplot(
  current_df,
  aes(x = U, y = modifier, linetype = U_ref, linewidth = U_ref)
) +
  geom_line() +
  scale_linetype_manual(
    values = ref_linetypes,
    name = "Reference velocity"
  ) +
  scale_linewidth_manual(
    values = ref_linewidths,
    guide = "none"
  ) +
  labs(
    x = expression("Mean current speed (m.s"^{-1}*")"),
    y = "Forcing modifier",
    title = "(a) Current forcing"
  ) +
  theme_classic() +
  plot_theme1 +
  forcing_theme

p_wave <- ggplot(
  wave_df,
  aes(x = U, y = modifier, linetype = U_ref, linewidth = U_ref)
) +
  geom_line() +
  scale_linetype_manual(
    values = ref_linetypes,
    name = "Reference velocity"
  ) +
  scale_linewidth_manual(
    values = ref_linewidths,
    guide = "none"
  ) +
  labs(
    x = expression("Seabed wave orbital velocity (m.s"^{-1}*")"),
    y = "Forcing modifier",
    title = "(b) Wave forcing"
  ) +
  theme_classic() +
  plot_theme1 +
  forcing_theme

shared_legend <- cowplot::get_legend(
  p_current +
    theme(
      legend.position = "bottom",
      legend.box.margin = margin(0, 0, 0, 0)
    )
)

figure_panels <- cowplot::plot_grid(
  p_current + theme(legend.position = "none"),
  p_wave + theme(legend.position = "none"),
  ncol = 2,
  align = "hv",
  axis = "tblr"
)

p_mechanical_response <- cowplot::plot_grid(
  figure_panels,
  shared_legend,
  ncol = 1,
  rel_heights = c(1, 0.10)
)

fig(7, 12)
p_mechanical_response


### Figure S7 — Monod response


In [ ]:
monod_modifier <- function(x, K = 1.0) {
  x / (x + K)
}

x_seq <- seq(0, 5, length.out = 300)

monod_df <- data.frame(
  x = rep(x_seq, 3),
  modifier = c(
    monod_modifier(x_seq, K = 0.5),
    monod_modifier(x_seq, K = 1.0),
    monod_modifier(x_seq, K = 2.0)
  ),
  K = factor(
    rep(c("0.5", "1.0 (baseline)", "2.0"), each = length(x_seq)),
    levels = c("0.5", "1.0 (baseline)", "2.0")
  )
)

p_monod <- ggplot(
  monod_df,
  aes(x, modifier, linetype = K, linewidth = K)
) +
  geom_line() +
  geom_hline(yintercept = 0.5, linetype = "dotted") +
  geom_vline(xintercept = 1.0, linetype = "dotted") +
  scale_linetype_manual(
    values = c(
      "0.5" = "dotted",
      "1.0 (baseline)" = "solid",
      "2.0" = "dashed"
    )
  ) +
  scale_linewidth_manual(
    values = c(
      "0.5" = 0.8,
      "1.0 (baseline)" = 1.2,
      "2.0" = 0.8
    ),
    guide = "none"
  ) +
  labs(
    x = "Chlorophyll concentration",
    y = "Biological modifier",
    linetype = "Half-saturation constant"
  ) +
  theme_classic() +
  plot_theme1

fig(7, 10)
p_monod


### Figure S8 — pH response


In [ ]:
gaussian_modifier <- function(x, x_ref = 8.1, sigma = 0.5) {
  exp(-0.5 * ((x - x_ref) / sigma)^2)
}

pH_seq <- seq(6.5, 9.0, length.out = 300)

ph_df <- data.frame(
  pH = rep(pH_seq, 3),
  modifier = c(
    gaussian_modifier(pH_seq, x_ref = 8.1, sigma = 0.25),
    gaussian_modifier(pH_seq, x_ref = 8.1, sigma = 0.50),
    gaussian_modifier(pH_seq, x_ref = 8.1, sigma = 0.75)
  ),
  sigma = factor(
    rep(c("0.25", "0.50 baseline", "0.75"), each = length(pH_seq)),
    levels = c("0.25", "0.50 baseline", "0.75")
  )
)

p_ph <- ggplot(
  ph_df,
  aes(pH, modifier, linetype = sigma, linewidth = sigma)
) +
  geom_line() +
  geom_hline(yintercept = 1, linetype = "dotted", colour = "grey40") +
  geom_vline(xintercept = 8.1, linetype = "dotted", colour = "grey40") +
  scale_linetype_manual(
    values = c(
      "0.25" = "dotted",
      "0.50 baseline" = "solid",
      "0.75" = "dashed"
    )
  ) +
  scale_linewidth_manual(
    values = c(
      "0.25" = 0.8,
      "0.50 baseline" = 1.2,
      "0.75" = 0.8
    ),
    guide = "none"
  ) +
  labs(
    x = "pH",
    y = "Relative biological modifier",
    linetype = "Response width (σ)"
  ) +
  theme_classic() +
  plot_theme1

fig(7, 10)
p_ph


### Figure S9 — Bivariate Teredo suitability response


In [ ]:
teredo_temp <- function(T_C, T_opt = 25, sigma_T = 5) {
  exp(-0.5 * ((T_C - T_opt) / sigma_T)^2)
}

teredo_salinity <- function(S, S_opt = 25, sigma_S = 8) {
  exp(-0.5 * ((S - S_opt) / sigma_S)^2)
}

teredo_grid <- expand.grid(
  salinity = seq(0, 40, length.out = 200),
  temperature = seq(0, 30, length.out = 200)
)

teredo_grid$suitability <-
  teredo_salinity(teredo_grid$salinity) *
  teredo_temp(teredo_grid$temperature)

p_teredo <- ggplot(teredo_grid) +
  
  geom_raster(
    aes(
      x = salinity,
      y = temperature,
      fill = suitability
    )
  ) +
  
  geom_contour(
    aes(
      x = salinity,
      y = temperature,
      z = suitability
    ),
    bins = 6,
    colour = "black",
    linewidth = 0.4,
    alpha = 0.6
  ) +

  scale_fill_gradient(
      low = "white",
      high = "#4F81BD", 
      name = 'Relative Teredo suitability',
      limits = c(0, 1),
      breaks = c(0, 0.25, 0.50, 0.75, 1)
  ) +

  guides(
    fill = guide_colourbar(
      title.position = "top",
      title.hjust = 0.5,
      barwidth = unit(12, "cm"),
      barheight = unit(0.5, "cm")
    )
  ) +
    
  coord_cartesian(
    xlim = c(5, 40.2),
    ylim = c(10, 30.2),
    expand = FALSE
  ) +
  
  scale_x_continuous(
    breaks = c(10, 20, 30)
  ) +

  labs(
    x = "Salinity",
    y = expression("Temperature (" * degree * "C)")
  ) +
  
  theme_classic() +
  annotate(
    "point",
    x = 25,
    y = 25,
    shape = 21,
    fill = "white",
    colour = "black",
    stroke = 0.5,
    size = 3
  ) +
  plot_theme1 

p_teredo


### Figure S10 — Cell-level sensitivity ratios


In [ ]:
cell <- fread(file.path(
  figure_data_dir,
  "supp_sensitivity_cell_ratios.csv"
))

cell[, value_plot := value]
cell[parameter == "Ea", value_plot := value_plot / 1000]

label_order <- c(
  "Current reference velocity",
  "Activation energy",
  "Teredo intensity",
  "Burial strength",
  "Burial exponent",
  "Wave reference velocity",
  "Salinity midpoint",
  "Current–wave interaction"
)

cell[, label := factor(
  parameter_labels[parameter],
  levels = label_order
)]

cell[, value_plot := factor(value_plot, levels = sort(unique(value_plot)))]

p_supp_sensitivity <- ggplot(cell) +
  geom_boxplot(
    aes(x = value_plot, y = ratio),
    outlier.shape = NA,
    width = 0.65
  ) +
  geom_hline(
    yintercept = 1,
    linetype = "dashed",
    linewidth = 0.4
  ) +
  facet_wrap(
    ~ label,
    scales = "free_x",
    ncol = 3
  ) +
  labs(
    x = NULL,
    y = "Relative rate scale"
  ) +
  theme_classic() +
  plot_theme1 +
  theme(
    strip.text = element_text(face = "bold"),
    panel.spacing = unit(1.0, "lines")
  )

fig(12, 12)
p_supp_sensitivity


### Figure S11 — Variance–mean relationships


In [ ]:
var_path <- file.path(ssd_dir, "variance_spatial_sample.csv")
vardat <- data.table::fread(var_path)

S11_theme <- theme(
  plot.title = element_text(
    size = 18,
    face = "bold",
    hjust = 0
  ),
  axis.title = element_text(
    size = 13
  ),
  axis.text = element_text(
    size = 10,
    colour = "black"
  ),
  plot.margin = margin(
    t = 8, r = 10, b = 8, l = 10
  )
)

S11a <- ggplot(
  vardat,
  aes(
    x = chl_mean,
    y = chl_var_total
  )
) +
  geom_point() +
  scale_x_log10(
    guide = "axis_logticks",
    labels = scales::label_number()
  ) +
  scale_y_log10(
    guide = "axis_logticks",
    labels = scales::label_number()
  ) +
  labs(
    title = "(a) Chlorophyll",
    x = expression("Chlorophyll mean (" * "log"[10] * ")"),
    y = expression("Chlorophyll variance (" * "log"[10] * ")")
  ) +
  theme_classic() +
  plot_theme1 +
  S11_theme

S11b <- ggplot(
  vardat,
  aes(
    x = VHM0_mean_mean,
    y = VHM0_p95_mean
  )
) +
  geom_point() +
  labs(
    title = "(b) Significant wave height",
    x = "Significant wave height median",
    y = "Significant wave height 95th percentile"
  ) +
  theme_classic() +
  plot_theme1 +
  S11_theme

S11c <- ggplot(
  vardat,
  aes(
    x = bottomT_mean,
    y = bottomT_var_total
  )
) +
  geom_point() +
  scale_x_log10(
    guide = "axis_logticks"
  ) +
  scale_y_log10(
    guide = "axis_logticks"
  ) +
  labs(
    title = "(c) Seafloor temperature",
    x = expression("Bottom temperature mean (" * "log"[10] * ")"),
    y = expression("Bottom temperature variance (" * "log"[10] * ")")
  ) +
  theme_classic() +
  plot_theme1 +
  S11_theme

S11d <- ggplot(
  vardat,
  aes(
    x = so_mean,
    y = so_var_total
  )
) +
  geom_point() +
  scale_x_log10(
    guide = "axis_logticks",
    labels = scales::label_number()
  ) +
  scale_y_log10(
    guide = "axis_logticks",
    labels = scales::label_number()
  ) +
  labs(
    title = "(d) Seafloor salinity",
    x = expression("Salinity mean (" * "log"[10] * ")"),
    y = expression("Salinity variance (" * "log"[10] * ")")
  ) +
  theme_classic() +
  plot_theme1 +
  S11_theme

fig_S11 <- cowplot::plot_grid(
  S11a,
  S11b,
  S11c,
  S11d,
  ncol = 2,
  align = "hv",
  axis = "tblr"
)

fig_S11


### Figure S12 — Temperature and salinity variance residuals


In [ ]:
variance_maps_dir <- file.path(
  ssd_dir,
  "variance_residual_maps"
)

bottomT_resid <- terra::rast(
  file.path(
    variance_maps_dir,
    "bottomT_var_total_residual_from_mean.tif"
  )
)

salinity_resid <- terra::rast(
  file.path(
    variance_maps_dir,
    "salinity_var_total_residual_from_mean.tif"
  )
)

S12_temp <- ggplot(world) +
  theme_bw() +
  tidyterra::geom_spatraster(
    data = bottomT_resid
  ) +
  geom_sf() +
  coord_sf(
    xlim = c(terra::xmin(bottomT_resid), terra::xmax(bottomT_resid)),
    ylim = c(terra::ymin(bottomT_resid), terra::ymax(bottomT_resid)),
    expand = FALSE
  ) +
  scale_fill_gradient2(
    low = "#3B83BD",
    mid = "#F7F7C6",
    high = "#D7301F",
    midpoint = 0,
    limits = c(-2.5, 2.5),
    oob = scales::squish,
    name = "Residual"
  ) +
  labs(
    title = "(a) Seafloor temperature residual"
  ) +
  theme(
    axis.text = element_text(
      size = 16,
      colour = "black"
    ),
    panel.background = element_rect(
      fill = "lightblue1",
      colour = "lightblue1"
    ),
    panel.grid.major = element_line(
      linewidth = 0.5,
      linetype = "solid",
      colour = "grey"
    ),
    plot.title = element_text(
      hjust = 0,
      face = "bold",
      size = 22
    ),
    legend.position = "none",
    plot.margin = margin(8, 5, 8, 5)
  )

S12_sal <- ggplot(world) +
  theme_bw() +
  tidyterra::geom_spatraster(
    data = salinity_resid
  ) +
  geom_sf() +
  coord_sf(
    xlim = c(terra::xmin(salinity_resid), terra::xmax(salinity_resid)),
    ylim = c(terra::ymin(salinity_resid), terra::ymax(salinity_resid)),
    expand = FALSE
  ) +
  scale_fill_gradient2(
    low = "#3B83BD",
    mid = "#F7F7C6",
    high = "#D7301F",
    midpoint = 0,
    limits = c(-2.5, 2.5),
    oob = scales::squish,
    name = "Residual"
  ) +
  labs(
    title = "(b) Salinity residual"
  ) +
  theme(
    axis.text = element_text(
      size = 16,
      colour = "black"
    ),
    panel.background = element_rect(
      fill = "lightblue1",
      colour = "lightblue1"
    ),
    panel.grid.major = element_line(
      linewidth = 0.5,
      linetype = "solid",
      colour = "grey"
    ),
    plot.title = element_text(
      hjust = 0,
      face = "bold",
      size = 22
    ),
    legend.position = "none",
    plot.margin = margin(8, 5, 8, 5)
  )

S12_legend_plot <- ggplot(
  data.frame(
    x = seq(-2.5, 2.5, length.out = 100),
    y = 1
  ),
  aes(x = x, y = y, fill = x)
) +
  geom_tile() +
  scale_fill_gradient2(
    low = "#3B83BD",
    mid = "#F7F7C6",
    high = "#D7301F",
    midpoint = 0,
    limits = c(-2.5, 2.5),
    name = "Residual"
  ) +
  guides(
    fill = guide_colourbar(
      title.position = "top",
      title.hjust = 0.5,
      barwidth = unit(12, "cm"),
      barheight = unit(0.5, "cm")
    )
  ) +
  theme_void() +
  theme(
    legend.position = "bottom"
  )

S12_legend <- cowplot::get_legend(
  S12_legend_plot
)

S12_maps <- cowplot::plot_grid(
  S12_temp,
  S12_sal,
  ncol = 1,
  align = "v",
  axis = "lr",
  rel_heights = c(1, 1)
)

fig_S12 <- cowplot::plot_grid(
  S12_maps,
  S12_legend,
  cowplot::ggdraw(),
  ncol = 1,
  rel_heights = c(1, 0.10, 0.03)
)

fig_S12


## 4. Export all publication figures


In [ ]:
# TIFF, 600 dpi, LZW compression.

save_tiff <- function(filename, plot, width, height) {
  ggsave(
    filename = file.path(figure_out_dir, filename),
    plot = plot,
    device = "tiff",
    width = width,
    height = height,
    units = "in",
    dpi = 600,
    compression = "lzw",
    bg = "white",
    limitsize = FALSE
  )
}

fig_S1 <- cowplot::plot_grid(
  p_age_class,
  p_wreck_counts,
  labels = c("(a)", "(b)"),
  label_size = 18,
  ncol = 2,
  align = "hv"
)

save_tiff("Figure_2.tiff", final_figure, 10, 7)
save_tiff("Figure_3.tiff", driver_maps, 12, 15)
save_tiff("Figure_4.tiff", p_sensitivity, 10, 7)
save_tiff("Figure_5.tiff", threshold_maps, 12, 15)
save_tiff("Figure_6.tiff", p_traj, 12, 10)

save_tiff("Figure_S1.tiff", fig_S1, 12, 6)
save_tiff("Figure_S2.tiff", p_basis, 9, 7)
save_tiff("Figure_S3.tiff", p_age_validation, 10, 7)
save_tiff("Figure_S4.tiff", p_arrhenius, 10, 7)
save_tiff("Figure_S5.tiff", p_salinity_response, 10, 7)
save_tiff("Figure_S6.tiff", p_mechanical_response, 12, 7)
save_tiff("Figure_S7.tiff", p_monod, 10, 7)
save_tiff("Figure_S8.tiff", p_ph, 10, 7)
save_tiff("Figure_S9.tiff", p_teredo, 10, 7)
save_tiff("Figure_S10.tiff", p_supp_sensitivity, 12, 12)
save_tiff("Figure_S11.tiff", fig_S11, 12, 10)
save_tiff("Figure_S12.tiff", fig_S12, 12, 12.8)

expected <- c(
  paste0("Figure_", 2:6, ".tiff"),
  paste0("Figure_S", 1:12, ".tiff")
)

written <- list.files(
  figure_out_dir,
  pattern = "\\.tiff$",
  full.names = FALSE
)

missing <- setdiff(expected, written)

cat("\nPublication figures written to:\n", figure_out_dir, "\n\n")

if (length(missing) == 0) {
  cat("All expected R-generated figures present:", length(expected), "files.\n")
} else {
  cat("MISSING:\n", paste(missing, collapse = "\n"), "\n")
}


## 5. Scope

This notebook generates publication figures only. Model simulations, spatial product generation and sensitivity calculations are performed upstream.
